<a href="https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fahad-Alam-Jamal/Flyrank_ML_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)
On Colab this clones the repo and installs requirements. Locally it just moves to the repo root.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Two paper findings + my methodology questions

Finding 1: The paper's Content Performance Curve says content peaks at 61-90 days, declines after 270 days, and older pages can recover after refresh. This appears to be based on age bucket comparisons using a 30-day trend-based health metric.

Methodology question: Where exactly does the label come from? I want to know whether the comparison is simply age bucket averages or whether it controls for demand and position, because older pages may already have different traffic profiles.

Second methodology question: Does the validation design support the recovery claim for 365+ pages? The paper notes that the 361+ bucket is small and unstable, so I would ask whether the sample size is enough to generalize this finding rather than describe a limited subset.

Finding 2: The paper's Freshness Multiplier says the 31-90 day window is the strongest stable freshness band and that 365+ day pages refreshed within 30 days show a large measured boost.

Methodology question: Is the label coming from observed growth ratios or from an actual refresh-action comparison? I would ask whether the analysis distinguishes pages that improved because they were refreshed from pages that were already on a positive trajectory.

Second methodology question: Does the validation design separate the effect of freshness from other correlated signals such as existing visibility or content age? I want to know if the paper is measuring an incremental refresh signal or simply describing the lifecycle of content with different visibility levels.


In [2]:
print('Section 1 notes are documented above. This code cell is a placeholder for any supporting checks.')


Section 1 notes are documented above. This code cell is a placeholder for any supporting checks.


## 2. My model under an honest split (before/after)

I rerun the Week-5 refresh model using a client-aware holdout split. This is the more honest validation for this problem because it keeps entire clients out of training and avoids leaking client-specific traffic and behavior patterns into the test set.

I compare the learned model to the Week-4 baseline on the same held-out clients, and I also note how a random row-level split differs for context.


In [3]:
import sys
from pathlib import Path

cwd = Path.cwd()
root = cwd
for parent in [cwd, *cwd.parents]:
    if (parent / "scripts").exists():
        root = parent
        break
sys.path.insert(0, str(root))

import os
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k

feature_path = root / 'data' / 'processed' / 'refresh_feature_vector.csv'
baseline_path = root / 'data' / 'processed' / 'baseline_refresh_queue.csv'

if not feature_path.exists():
    os.system(f'{sys.executable} {root / "scripts" / "01_prepare_features.py"}')
if not baseline_path.exists():
    os.system(f'{sys.executable} {root / "scripts" / "02_baseline_score.py"}')

features = pd.read_csv(feature_path)
baseline_queue = pd.read_csv(baseline_path)

numeric_features = MODEL_NUMERIC_FEATURES
categorical_features = MODEL_CATEGORICAL_FEATURES

model_features = features[numeric_features + categorical_features].copy()
model_features[numeric_features] = model_features[numeric_features].apply(pd.to_numeric, errors='coerce').fillna(0)
model_features[categorical_features] = model_features[categorical_features].fillna('unknown').astype(str)
target = features['is_declining_label'].astype(int)

client_series = features['client_id'].fillna('unknown').astype(str)
clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(42)
shuffled_clients = rng.permutation(clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = client_series.isin(test_clients).to_numpy()
train_mask = ~test_mask

X_train = model_features.iloc[train_mask]
X_test = model_features.iloc[test_mask]
y_train = target.iloc[train_mask]
y_test = target.iloc[test_mask]

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value=0)), ('scale', StandardScaler())]), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ],
    remainder='drop',
)

model = Pipeline([
    ('preprocess', preprocess),
    ('classifier', LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=5000, random_state=42)),
])

model.fit(X_train, y_train)
model_probs = model.predict_proba(X_test)[:, 1]
baseline_lookup = baseline_queue.set_index('content_id')['baseline_refresh_score']
baseline_scores = features.loc[test_mask, 'content_id'].map(baseline_lookup).fillna(0).to_numpy()

comparison = pd.DataFrame([
    ('precision_at_20', precision_at_k(y_test, baseline_scores, 20), precision_at_k(y_test, model_probs, 20)),
    ('precision_at_50', precision_at_k(y_test, baseline_scores, 50), precision_at_k(y_test, model_probs, 50)),
    ('precision_at_100', precision_at_k(y_test, baseline_scores, 100), precision_at_k(y_test, model_probs, 100)),
    ('roc_auc', float(roc_auc_score(y_test, baseline_scores)), float(roc_auc_score(y_test, model_probs))),
    ('average_precision', float(average_precision_score(y_test, baseline_scores)), float(average_precision_score(y_test, model_probs))),
], columns=['metric', 'baseline', 'model'])
comparison['delta'] = comparison['model'] - comparison['baseline']

print('Client-aware holdout comparison')
print(comparison.round(3).to_string(index=False))
print()
print('Held-out rows:', len(X_test))
print('Held-out clients:', len(test_clients))
print('Base decline rate:', round(float(y_test.mean()), 3))

from sklearn.model_selection import train_test_split
X_rand_train, X_rand_test, y_rand_train, y_rand_test = train_test_split(model_features, target, test_size=0.2, random_state=42, stratify=target)
model_rand = Pipeline([('preprocess', preprocess), ('classifier', LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=5000, random_state=42))])
model_rand.fit(X_rand_train, y_rand_train)
probs_rand = model_rand.predict_proba(X_rand_test)[:, 1]
print()
print('Random split context metrics:')
print('precision_at_20', precision_at_k(y_rand_test, probs_rand, 20), 'precision_at_50', precision_at_k(y_rand_test, probs_rand, 50), 'precision_at_100', precision_at_k(y_rand_test, probs_rand, 100))
print('roc_auc', roc_auc_score(y_rand_test, probs_rand), 'avg_prec', average_precision_score(y_rand_test, probs_rand))

test_frame = features.loc[test_mask].copy()
test_frame['baseline_score'] = baseline_scores
test_frame['model_probability'] = model_probs
test_frame['true_label'] = y_test.to_numpy()
test_frame['model_prediction'] = (test_frame['model_probability'] >= 0.5).astype(int)
test_frame['error_type'] = np.where((test_frame['model_prediction'] == 1) & (test_frame['true_label'] == 0), 'false_positive', np.where((test_frame['model_prediction'] == 0) & (test_frame['true_label'] == 1), 'false_negative', 'correct'))

print()
print('Top false positives by model confidence:')
print(test_frame.loc[test_frame['error_type'] == 'false_positive']
    .sort_values('model_probability', ascending=False)
    .head(5)[['content_id', 'impressions_90d', 'days_since_last_update', 'avg_position', 'ctr', 'model_probability', 'baseline_score']]
    .to_string(index=False))
print()
print('Top false negatives by low model confidence:')
print(test_frame.loc[test_frame['error_type'] == 'false_negative']
    .sort_values('model_probability', ascending=True)
    .head(5)[['content_id', 'impressions_90d', 'days_since_last_update', 'avg_position', 'ctr', 'model_probability', 'baseline_score']]
    .to_string(index=False))


Client-aware holdout comparison
           metric  baseline  model  delta
  precision_at_20     0.150  0.350  0.200
  precision_at_50     0.240  0.400  0.160
 precision_at_100     0.360  0.440  0.080
          roc_auc     0.627  0.702  0.075
average_precision     0.468  0.523  0.056

Held-out rows: 2325
Held-out clients: 6
Base decline rate: 0.391

Random split context metrics:
precision_at_20 0.9 precision_at_50 0.9 precision_at_100 0.89
roc_auc 0.7107380789965105 avg_prec 0.727105925415965

Top false positives by model confidence:
          content_id  impressions_90d  days_since_last_update  avg_position  ctr  model_probability  baseline_score
content_8fdbff16a886             3863                      20          12.0 0.00           0.895541        0.559004
content_e354f8e518c2             1163                      20          23.4 0.00           0.877226        0.413314
content_da806b9f243e             2268                      20           5.8 0.18           0.876895        0.5319

## 3. Leakage audit

I audit the final feature set for leakage and future-derived signals. The model uses observable current counts, positions, and engagement metrics only. It does not include `trend_direction` or any direct label field in the input features.

This means the main risk is not a direct leakage bug, but the usual observational issue: the label is derived from the same snapshot as the features, so the findings are best framed as decision-support evidence rather than causal proof.


In [4]:
import sys
from pathlib import Path

cwd = Path.cwd()
root = cwd
for parent in [cwd, *cwd.parents]:
    if (parent / "scripts").exists():
        root = parent
        break
sys.path.insert(0, str(root))

import pandas as pd
from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

features = pd.read_csv(root / "data" / "processed" / "refresh_feature_vector.csv")
model_feature_names = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
leakage_candidates = [name for name in model_feature_names if any(token in name.lower() for token in ['trend', 'label', 'future', 'next'])]
print('Model feature count:', len(model_feature_names))
print('Leakage candidate feature names:', leakage_candidates)
print('Includes trend_direction?', 'trend_direction' in model_feature_names)
print('Includes is_declining_label?', 'is_declining_label' in model_feature_names)

client_series = features['client_id'].fillna('unknown').astype(str)
clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(42)
shuffled = rng.permutation(clients)
test_clients = set(shuffled[:max(1, int(round(len(shuffled) * 0.2)))])
train_mask = ~client_series.isin(test_clients).to_numpy()
test_mask = client_series.isin(test_clients).to_numpy()
train_content = set(features.loc[train_mask, 'content_id'])
test_content = set(features.loc[test_mask, 'content_id'])
print('Content overlap train/test:', len(train_content & test_content))


Model feature count: 26
Leakage candidate feature names: []
Includes trend_direction? False
Includes is_declining_label? False
Content overlap train/test: 0


## 4. Claim rewrite

Original bold claim: "The learned model is clearly better than the Week-4 rule and should replace it for refresh prioritization."

Rewritten safe claim: "On a client-aware holdout split, the logistic refresh model observed higher precision than the Week-4 rule baseline. This measured improvement suggests it may provide stronger decision-support evidence for ranking refresh candidates, but it is based on this snapshot and not a guarantee of future outcomes."

This language keeps the emphasis on observation, measurement, and decision support rather than on absolute superiority or causal certainty.


In [5]:
original_claim = 'The learned model is clearly better than the Week-4 rule and should replace it for refresh prioritization.'
rewritten_claim = ('On a client-aware holdout split, the logistic refresh model observed higher precision than the Week-4 rule baseline. '                 'This measured improvement suggests it may provide stronger decision-support evidence for ranking refresh candidates, '                 'but it is based on this snapshot and not a guarantee of future outcomes.')
print('Original claim:')
print(original_claim)
print()
print('Rewritten claim:')
print(rewritten_claim)


Original claim:
The learned model is clearly better than the Week-4 rule and should replace it for refresh prioritization.

Rewritten claim:
On a client-aware holdout split, the logistic refresh model observed higher precision than the Week-4 rule baseline. This measured improvement suggests it may provide stronger decision-support evidence for ranking refresh candidates, but it is based on this snapshot and not a guarantee of future outcomes.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.